# Exploración de datos de Instacart con Python y Pandas

**Actividad:** GA1-220501094-01-AA1-EV04 – Fundamentos en Pandas · *Explorando Instacart: descubre patrones de compra*
**Rol:** analista de datos de Instacart (desarrollo individual)
**Herramientas:** Python · Pandas · Jupyter Notebook

**Contenido**

1. Momento 1 · Comprensión del problema y reconocimiento de los datos
2. Momento 2 · Importación y exploración inicial de los archivos
3. Momento 3 · Estructura y calidad inicial de los datos
4. Momento 4 · Indexación y filtrado lógico
5. Momento 5 · Indicadores básicos
6. Momento 6 · Resolución del problema y síntesis

> **Cómo ejecutar:** coloque los cinco CSV en la carpeta `data/` de la raíz del proyecto (este notebook está en `notebooks/` y los lee desde `../data/`) y use *Run All*. Todas las cifras de las interpretaciones se calculan a partir de los datos, no están escritas a mano.

---
# Momento 1 · Comprensión del problema y reconocimiento de los datos

## 1.1 Descripción del problema

Instacart es una plataforma de compras de supermercado. El equipo que analiza sus operaciones necesita saber **qué información contienen** los archivos que la plataforma registra (pedidos, productos, pasillos, departamentos y artículos incluidos en cada compra) y **cómo consultarla** para responder preguntas sobre el comportamiento de los clientes.

Como analista de datos, mi tarea es hacer una **exploración inicial**: identificar la estructura de los datos, consultar registros específicos y obtener indicadores básicos. En esta fase **no se limpian ni se modifican** los datos; solo se documentan sus características y limitaciones.

**Pregunta orientadora:** ¿cómo utilizar Python y Pandas para explorar los datos de Instacart, identificar las características de los pedidos y productos, y obtener información relevante sobre los hábitos de compra de sus clientes?

## 1.2 Los cinco archivos y la información que almacenan

| N.º | Archivo | Qué información considero que almacena |
|---|---|---|
| 1 | `instacart_orders.csv` | Un registro por pedido: quién lo hizo (`user_id`), cuántos pedidos lleva ese usuario (`order_number`), día de la semana y hora de compra, y días transcurridos desde el pedido anterior. |
| 2 | `products.csv` | Catálogo de productos: identificador, nombre y su clasificación en pasillo y departamento. |
| 3 | `aisles.csv` | Tabla de consulta que traduce `aisle_id` al nombre del pasillo. |
| 4 | `departments.csv` | Tabla de consulta que traduce `department_id` al nombre del departamento. |
| 5 | `order_products.csv` | Detalle de cada pedido: qué productos incluye, en qué posición se agregó cada uno al carrito y si es una recompra. |

## 1.3 Columnas que identifican pedidos, usuarios y productos

| Entidad | Columna | Dónde aparece |
|---|---|---|
| **Pedido** | `order_id` | `orders` (identifica cada pedido) y `order_products` (enlace con los artículos) |
| **Usuario** | `user_id` | `orders` |
| **Producto** | `product_id` | `products` (identifica cada producto) y `order_products` (enlace con los pedidos) |
| Pasillo | `aisle_id` | `products` y `aisles` |
| Departamento | `department_id` | `products` y `departments` |

## 1.4 Tres preguntas de análisis

1. **¿A qué horas del día y en qué días de la semana se concentran los pedidos?** Se responde contando pedidos con `order_hour_of_day` y `order_dow`.
2. **¿Con qué frecuencia vuelven a comprar los usuarios?** Se responde con el promedio de `days_since_prior_order` y de `order_number`.
3. **¿Qué proporción de los artículos comprados son recompras, y cuántos productos se agregan primero al carrito?** Se responde con `reordered` y `add_to_cart_order`.

## 1.5 ¿Por qué explorar los datos antes de analizarlos?

Explorar primero permite saber **cuántos datos hay, de qué tipo son y qué tan completos están**. Sin ese reconocimiento se corre el riesgo de interpretar mal un promedio por valores ausentes, de contar como "clientes" lo que en realidad son filas, o de cruzar tablas por columnas equivocadas. Una exploración inicial documenta las limitaciones de los datos *antes* de sacar conclusiones, de modo que los resultados posteriores sean confiables y reproducibles.

---
# Momento 2 · Importación y exploración inicial de los archivos

## 2.1 Importación de la librería y carga de los archivos

Los archivos usan **punto y coma (`;`)** como separador, por eso se indica `sep=";"` en `read_csv()`.

In [1]:
import os
import pandas as pd
from IPython.display import display, Markdown

# Carpeta con los CSV (ruta relativa, funciona en Windows, Linux y macOS)
if os.path.isdir("../data"):
    RUTA = "../data/"  # el notebook está en notebooks/ y los datos en la raíz
elif os.path.isdir("data"):
    RUTA = "data/"
else:
    RUTA = ""  # p. ej. Google Colab, con los CSV subidos junto al notebook

orders = pd.read_csv(RUTA + "instacart_orders.csv", sep=";")
products = pd.read_csv(RUTA + "products.csv", sep=";")
aisles = pd.read_csv(RUTA + "aisles.csv", sep=";")
departments = pd.read_csv(RUTA + "departments.csv", sep=";")
order_products = pd.read_csv(RUTA + "order_products.csv", sep=";")

# Diccionario auxiliar para recorrer los cinco DataFrames
datasets = {
    "orders": orders,
    "products": products,
    "aisles": aisles,
    "departments": departments,
    "order_products": order_products,
}


def fmt(x, dec=0):
    """Formato colombiano: punto para miles y coma para decimales."""
    s = f"{x:,.{dec}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")


print("Pandas", pd.__version__)
print("Archivos cargados:", ", ".join(datasets))

Pandas 3.1.0
Archivos cargados: orders, products, aisles, departments, order_products


## 2.2 Primeros y últimos cinco registros (`head()` y `tail()`)

In [2]:
print("orders · primeros 5 registros")
display(orders.head())
print("orders · últimos 5 registros")
display(orders.tail())

orders · primeros 5 registros


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
1,1690866,163593,5,5,12,9.0
2,1454967,39980,4,5,19,2.0
3,1768857,82516,56,0,20,10.0
4,3007858,196724,2,4,12,17.0


orders · últimos 5 registros


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
478962,3210681,5617,5,1,14,7.0
478963,3270802,112087,2,3,13,6.0
478964,885349,82944,16,2,11,6.0
478965,216274,4391,3,3,8,8.0
478966,2071924,1730,18,1,14,15.0


In [3]:
print("products · primeros 5 registros")
display(products.head())
print("products · últimos 5 registros")
display(products.tail())

products · primeros 5 registros


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


products · últimos 5 registros


,product_id,product_name,aisle_id,department_id
49689,49690,HIGH PERFORMANCE ENERGY DRINK,64,7
49690,49691,ORIGINAL PANCAKE & WAFFLE MIX,130,14
49691,49692,ORGANIC INSTANT OATMEAL LIGHT MAPLE BROWN SUGAR,130,14
49692,49693,SPRING WATER BODY WASH,127,11
49693,49694,BURRITO- STEAK & CHEESE,38,1


In [4]:
print("aisles · primeros 5 registros")
display(aisles.head())
print("aisles · últimos 5 registros")
display(aisles.tail())

aisles · primeros 5 registros


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


aisles · últimos 5 registros


,aisle_id,aisle
129,130,hot cereal pancake mixes
130,131,dry pasta
131,132,beauty
132,133,muscles joints pain relief
133,134,specialty wines champagnes


In [5]:
print("departments · primeros 5 registros")
display(departments.head())
print("departments · últimos 5 registros")
display(departments.tail())

departments · primeros 5 registros


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


departments · últimos 5 registros


,department_id,department
16,17,household
17,18,babies
18,19,snacks
19,20,deli
20,21,missing


In [6]:
print("order_products · primeros 5 registros")
display(order_products.head())
print("order_products · últimos 5 registros")
display(order_products.tail())

order_products · primeros 5 registros


,order_id,product_id,add_to_cart_order,reordered
0,2141543,11440,17.0,0
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
3,491251,8670,35.0,1
4,2571142,1940,5.0,1


order_products · últimos 5 registros


,order_id,product_id,add_to_cart_order,reordered
4545002,577211,15290,12.0,1
4545003,1219554,21914,9.0,0
4545004,692640,47766,4.0,1
4545005,319435,691,8.0,1
4545006,1398151,28733,9.0,0


## 2.3 Dimensiones de cada DataFrame (`shape`)

In [7]:
dimensiones = pd.DataFrame(
    {
        "filas": {n: df.shape[0] for n, df in datasets.items()},
        "columnas": {n: df.shape[1] for n, df in datasets.items()},
    }
)
dimensiones.index.name = "DataFrame"

display(dimensiones.style.format({"filas": lambda v: fmt(v), "columnas": lambda v: fmt(v)}))

# Comparación con las cifras indicadas en el enunciado de la actividad
esperado = {
    "orders": (478967, 6),
    "products": (49694, 4),
    "aisles": (134, 2),
    "departments": (21, 2),
    "order_products": (4545007, 4),
}
for n, df in datasets.items():
    estado = "coincide" if df.shape == esperado[n] else f"DIFIERE (esperado {esperado[n]})"
    print(f"{n:15s} {df.shape} -> {estado} con el enunciado")

,filas,columnas
DataFrame,,
orders,478.967,6
products,49.694,4
aisles,134,2
departments,21,2
order_products,4.545.007,4


orders          (478967, 6) -> coincide con el enunciado
products        (49694, 4) -> coincide con el enunciado
aisles          (134, 2) -> coincide con el enunciado
departments     (21, 2) -> coincide con el enunciado
order_products  (4545007, 4) -> coincide con el enunciado


## 2.4 Preguntas de análisis

In [8]:
mas_filas = dimensiones["filas"].idxmax()
min_cols = dimensiones["columnas"].min()
menos_columnas = ", ".join(dimensiones.index[dimensiones["columnas"] == min_cols])

display(Markdown(f"""
**¿Cuál de los cinco DataFrames contiene más registros?**
`{mas_filas}`, con **{fmt(dimensiones.loc[mas_filas, 'filas'])}** filas. Contiene un registro por cada producto incluido en un pedido, por eso supera ampliamente a los demás.

**¿Cuál contiene menos columnas?**
{menos_columnas}, con **{min_cols}** columnas cada uno (un identificador y un nombre). Son tablas de consulta (*lookup*).

**¿Qué diferencias identifico entre los datos de pedidos y los de productos?**
- `orders` tiene **{fmt(orders.shape[0])}** filas y **{orders.shape[1]}** columnas; cada fila es un *pedido* y describe **cuándo** y **quién** compró (día, hora, usuario, frecuencia).
- `products` tiene **{fmt(products.shape[0])}** filas y **{products.shape[1]}** columnas; cada fila es un *producto* del catálogo y describe **qué** se vende y cómo se clasifica.
- Los pedidos son datos transaccionales (crecen con cada compra); los productos son un catálogo relativamente estable.
- Se conectan a través de `order_products`, que enlaza `order_id` con `product_id`.

**¿Por qué es importante conocer las dimensiones de un conjunto de datos?**
Porque define cuánta memoria y tiempo exige procesarlo (aquí `order_products` es el más pesado), permite verificar que la carga fue completa (contra lo esperado), y ayuda a detectar errores como filas faltantes o columnas mal separadas.
"""))


**¿Cuál de los cinco DataFrames contiene más registros?**
`order_products`, con **4.545.007** filas. Contiene un registro por cada producto incluido en un pedido, por eso supera ampliamente a los demás.

**¿Cuál contiene menos columnas?**
aisles, departments, con **2** columnas cada uno (un identificador y un nombre). Son tablas de consulta (*lookup*).

**¿Qué diferencias identifico entre los datos de pedidos y los de productos?**
- `orders` tiene **478.967** filas y **6** columnas; cada fila es un *pedido* y describe **cuándo** y **quién** compró (día, hora, usuario, frecuencia).
- `products` tiene **49.694** filas y **4** columnas; cada fila es un *producto* del catálogo y describe **qué** se vende y cómo se clasifica.
- Los pedidos son datos transaccionales (crecen con cada compra); los productos son un catálogo relativamente estable.
- Se conectan a través de `order_products`, que enlaza `order_id` con `product_id`.

**¿Por qué es importante conocer las dimensiones de un conjunto de datos?**
Porque define cuánta memoria y tiempo exige procesarlo (aquí `order_products` es el más pesado), permite verificar que la carga fue completa (contra lo esperado), y ayuda a detectar errores como filas faltantes o columnas mal separadas.


---
# Momento 3 · Reconocimiento de la estructura y calidad inicial de los datos

## 3.1 Nombres de columnas (`columns`)

In [9]:
for n, df in datasets.items():
    print(f"{n:15s} {list(df.columns)}")

orders          ['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
products        ['product_id', 'product_name', 'aisle_id', 'department_id']
aisles          ['aisle_id', 'aisle']
departments     ['department_id', 'department']
order_products  ['order_id', 'product_id', 'add_to_cart_order', 'reordered']


## 3.2 Tipos de datos (`dtypes`) y resumen con `info()`

Primero se consulta el atributo `dtypes` de cada DataFrame, que muestra el tipo de cada columna. Luego `info(show_counts=True)` amplía ese resumen con el conteo de no nulos y el uso de memoria; `show_counts=True` fuerza a mostrar el conteo aun en tablas muy grandes como `order_products`.

In [10]:
for n, df in datasets.items():
    print(n)
    print(df.dtypes, "\n")

orders
order_id                    int64
user_id                     int64
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object 

products
product_id       int64
product_name       str
aisle_id         int64
department_id    int64
dtype: object 

aisles
aisle_id    int64
aisle         str
dtype: object 

departments
department_id    int64
department         str
dtype: object 

order_products
order_id               int64
product_id             int64
add_to_cart_order    float64
reordered              int64
dtype: object 



In [11]:
print("orders")
orders.info(show_counts=True)

orders


<class 'pandas.DataFrame'>
RangeIndex: 478967 entries, 0 to 478966
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   order_id                478967 non-null  int64  
 1   user_id                 478967 non-null  int64  
 2   order_number            478967 non-null  int64  
 3   order_dow               478967 non-null  int64  
 4   order_hour_of_day       478967 non-null  int64  
 5   days_since_prior_order  450148 non-null  float64
dtypes: float64(1), int64(5)
memory usage: 21.9 MB


In [12]:
print("products")
products.info(show_counts=True)

products
<class 'pandas.DataFrame'>
RangeIndex: 49694 entries, 0 to 49693
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   product_id     49694 non-null  int64
 1   product_name   48436 non-null  str  
 2   aisle_id       49694 non-null  int64
 3   department_id  49694 non-null  int64
dtypes: int64(3), str(1)
memory usage: 1.5 MB


In [13]:
print("aisles")
aisles.info(show_counts=True)

aisles
<class 'pandas.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   aisle_id  134 non-null    int64
 1   aisle     134 non-null    str  
dtypes: int64(1), str(1)
memory usage: 2.2 KB


In [14]:
print("departments")
departments.info(show_counts=True)

departments
<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   department_id  21 non-null     int64
 1   department     21 non-null     str  
dtypes: int64(1), str(1)
memory usage: 468.0 bytes


In [15]:
print("order_products")
order_products.info(show_counts=True)

order_products
<class 'pandas.DataFrame'>
RangeIndex: 4545007 entries, 0 to 4545006
Data columns (total 4 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   order_id           4545007 non-null  int64  
 1   product_id         4545007 non-null  int64  
 2   add_to_cart_order  4544171 non-null  float64
 3   reordered          4545007 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 138.7 MB


In [16]:
def categoria_tipo(serie):
    if pd.api.types.is_integer_dtype(serie):
        return "entero"
    if pd.api.types.is_float_dtype(serie):
        return "decimal"
    return "texto"


tipos = pd.DataFrame(
    [
        {"DataFrame": n, "columna": c, "dtype": str(df[c].dtype), "categoría": categoria_tipo(df[c])}
        for n, df in datasets.items()
        for c in df.columns
    ]
)
display(tipos)

resumen_tipos = tipos.groupby(["DataFrame", "categoría"]).size().unstack(fill_value=0)
display(resumen_tipos)

,DataFrame,columna,dtype,categoría
0,orders,order_id,int64,entero
1,orders,user_id,int64,entero
2,orders,order_number,int64,entero
3,orders,order_dow,int64,entero
4,orders,order_hour_of_day,int64,entero
5,orders,days_since_prior_order,float64,decimal
6,products,product_id,int64,entero
7,products,product_name,str,texto
8,products,aisle_id,int64,entero
9,products,department_id,int64,entero


categoría,decimal,entero,texto
DataFrame,,,
aisles,0,1,1
departments,0,1,1
order_products,1,3,0
orders,1,5,0
products,0,3,1


## 3.3 Campos que relacionan conceptualmente los conjuntos de datos

| Relación | Campo común |
|---|---|
| `orders` ↔ `order_products` | `order_id` |
| `products` ↔ `order_products` | `product_id` |
| `products` ↔ `aisles` | `aisle_id` |
| `products` ↔ `departments` | `department_id` |

`order_products` actúa como tabla puente entre pedidos y productos; `aisles` y `departments` solo traducen identificadores a nombres.

## 3.4 Valores no nulos frente a registros (valores ausentes)

Comparo el número de filas (`shape`) con el conteo de no nulos que reporta `info()`. La diferencia son valores ausentes.

In [17]:
filas_ausentes = []
for n, df in datasets.items():
    for c in df.columns:
        faltan = int(df[c].isna().sum())
        filas_ausentes.append(
            {
                "DataFrame": n,
                "columna": c,
                "filas": df.shape[0],
                "no nulos": int(df[c].notna().sum()),
                "ausentes": faltan,
                "% ausente": round(100 * faltan / df.shape[0], 3),
            }
        )
ausentes = pd.DataFrame(filas_ausentes)

print("Columnas con valores ausentes:")
display(ausentes[ausentes["ausentes"] > 0].reset_index(drop=True))

# Verificación: ¿los ausentes de days_since_prior_order corresponden al primer pedido?
na_dias = orders["days_since_prior_order"].isna()
if na_dias.any():
    todos_primer_pedido = bool((orders.loc[na_dias, "order_number"] == 1).all())
    print(
        f"\ndays_since_prior_order ausente en {fmt(na_dias.sum())} pedidos; "
        f"¿todos tienen order_number == 1? {todos_primer_pedido}"
    )

Columnas con valores ausentes:


,DataFrame,columna,filas,no nulos,ausentes,% ausente
0,orders,days_since_prior_order,478967,450148,28819,6.017
1,products,product_name,49694,48436,1258,2.531
2,order_products,add_to_cart_order,4545007,4544171,836,0.018



days_since_prior_order ausente en 28.819 pedidos; ¿todos tienen order_number == 1? True


## 3.5 Tabla de diagnóstico

Además de los ausentes, reviso filas completamente duplicadas e identificadores repetidos en las columnas que deberían ser únicas.

In [18]:
claves = {
    "orders": ["order_id"],
    "products": ["product_id"],
    "aisles": ["aisle_id"],
    "departments": ["department_id"],
    "order_products": ["order_id", "product_id"],
}

filas_diag = []
for n, df in datasets.items():
    partes = []

    con_nulos = df.isna().sum()
    con_nulos = con_nulos[con_nulos > 0]
    if len(con_nulos):
        partes.append(
            "Valores ausentes en "
            + ", ".join(f"{c} ({fmt(v)})" for c, v in con_nulos.items())
        )
    else:
        partes.append("Sin valores ausentes")

    partes.append(f"{fmt(df.duplicated().sum())} filas completamente duplicadas")
    k = claves[n]
    partes.append(
        f"{fmt(df.duplicated(subset=k).sum())} repetidos en la clave ({', '.join(k)})"
    )

    filas_diag.append(
        {
            "archivo": n + ".csv",
            "filas": fmt(df.shape[0]),
            "columnas": df.shape[1],
            "observaciones": ". ".join(partes) + ".",
        }
    )

diagnostico = pd.DataFrame(filas_diag)
with pd.option_context("display.max_colwidth", 200):
    display(diagnostico)

,archivo,filas,columnas,observaciones
0,orders.csv,478.967,6,Valores ausentes en days_since_prior_order (28.819). 15 filas completamente duplicadas. 15 repetidos en la clave (order_id).
1,products.csv,49.694,4,Valores ausentes en product_name (1.258). 0 filas completamente duplicadas. 0 repetidos en la clave (product_id).
2,aisles.csv,134,2,Sin valores ausentes. 0 filas completamente duplicadas. 0 repetidos en la clave (aisle_id).
3,departments.csv,21,2,Sin valores ausentes. 0 filas completamente duplicadas. 0 repetidos en la clave (department_id).
4,order_products.csv,4.545.007,4,"Valores ausentes en add_to_cart_order (836). 0 filas completamente duplicadas. 0 repetidos en la clave (order_id, product_id)."


## 3.6 Preguntas de análisis

In [19]:
cols_ausentes = ausentes[ausentes["ausentes"] > 0]
if len(cols_ausentes):
    lista_ausentes = "; ".join(
        f"`{r.DataFrame}.{r.columna}` ({fmt(r.ausentes)} ausentes, {fmt(r['% ausente'], 2)} %)"
        for _, r in cols_ausentes.iterrows()
    )
else:
    lista_ausentes = "no se observan valores ausentes"

display(Markdown(f"""
**¿Qué diferencias existen entre `int64`, `float64` y `object`?**
- `int64`: números enteros (identificadores, conteos, horas). No admite `NaN`.
- `float64`: números con decimales; también es el tipo que usa Pandas para una columna entera que contiene ausentes, porque `NaN` es un decimal.
- `object` (en Pandas 3 aparece como `str`): texto, como `product_name`, `aisle` y `department`.

**¿Qué información proporciona `info()` que no se obtiene con `shape`?**
`shape` solo da filas y columnas. `info()` añade el **tipo de cada columna**, el **conteo de no nulos** (que permite detectar ausentes) y el **uso de memoria**.

**¿En qué columnas se observan valores ausentes?**
{lista_ausentes}.

**¿Qué dificultades podrían generar estos valores en un análisis posterior?**
- Los promedios y estadísticas ignoran los `NaN`, por lo que se calculan sobre menos registros que el total.
- Un filtro como `add_to_cart_order == 1` nunca selecciona filas con valor ausente.
- Un producto sin nombre dificulta reportes por producto; un cruce (`merge`) puede dejar campos vacíos.
- Una columna entera con ausentes pasa a `float64`, lo que puede confundir.
- Los identificadores repetidos (si existen) inflan los conteos de pedidos o registros.
"""))


**¿Qué diferencias existen entre `int64`, `float64` y `object`?**
- `int64`: números enteros (identificadores, conteos, horas). No admite `NaN`.
- `float64`: números con decimales; también es el tipo que usa Pandas para una columna entera que contiene ausentes, porque `NaN` es un decimal.
- `object` (en Pandas 3 aparece como `str`): texto, como `product_name`, `aisle` y `department`.

**¿Qué información proporciona `info()` que no se obtiene con `shape`?**
`shape` solo da filas y columnas. `info()` añade el **tipo de cada columna**, el **conteo de no nulos** (que permite detectar ausentes) y el **uso de memoria**.

**¿En qué columnas se observan valores ausentes?**
`orders.days_since_prior_order` (28.819 ausentes, 6,02 %); `products.product_name` (1.258 ausentes, 2,53 %); `order_products.add_to_cart_order` (836 ausentes, 0,02 %).

**¿Qué dificultades podrían generar estos valores en un análisis posterior?**
- Los promedios y estadísticas ignoran los `NaN`, por lo que se calculan sobre menos registros que el total.
- Un filtro como `add_to_cart_order == 1` nunca selecciona filas con valor ausente.
- Un producto sin nombre dificulta reportes por producto; un cruce (`merge`) puede dejar campos vacíos.
- Una columna entera con ausentes pasa a `float64`, lo que puede confundir.
- Los identificadores repetidos (si existen) inflan los conteos de pedidos o registros.


---
# Momento 4 · Indexación y filtrado lógico de la información

Se usa `loc[]` con **máscaras booleanas**: `df.loc[condición, columnas]`. En condiciones simultáneas cada comparación va entre paréntesis y se une con `&` (y) o `|` (o).

Para no saturar la pantalla, cada ejercicio muestra los primeros registros y el total de filas obtenido.

## Requerimiento 1 · `product_id` y `product_name` (products)

In [20]:
ej1 = products.loc[:, ["product_id", "product_name"]]
display(ej1.head())
print(f"Resultado: {fmt(ej1.shape[0])} filas x {ej1.shape[1]} columnas")

,product_id,product_name
0,1,Chocolate Sandwich Cookies
1,2,All-Seasons Salt
2,3,Robust Golden Unsweetened Oolong Tea
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...
4,5,Green Chile Anytime Sauce


Resultado: 49.694 filas x 2 columnas


**Explicación:** seleccionar columnas conserva todas las filas y reduce las columnas a las dos pedidas. Sirve para trabajar solo con la información necesaria.

## Requerimiento 2 · productos con `department_id` igual a 1 (products)

In [21]:
ej2 = products.loc[products["department_id"] == 1]
display(ej2.head())
print(f"Resultado: {fmt(ej2.shape[0])} de {fmt(products.shape[0])} productos ({fmt(100 * ej2.shape[0] / products.shape[0], 2)} %)")

,product_id,product_name,aisle_id,department_id
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
7,8,Cut Russet Potatoes Steam N' Mash,116,1
11,12,Chocolate Fudge Layer Cake,119,1
17,18,Pizza for One Suprema Frozen Pizza,79,1
29,30,Three Cheese Ziti- Marinara with Meatballs,38,1


Resultado: 4.008 de 49.694 productos (8,07 %)


**Explicación:** el filtro conserva las columnas y deja solo las filas del departamento 1; el resultado es un subconjunto del catálogo.

## Requerimiento 3 · pedidos realizados a las 10 horas (orders)

In [22]:
ej3 = orders.loc[orders["order_hour_of_day"] == 10]
display(ej3.head())
print(f"Resultado: {fmt(ej3.shape[0])} de {fmt(orders.shape[0])} pedidos ({fmt(100 * ej3.shape[0] / orders.shape[0], 2)} %)")

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
19,294269,107523,29,2,10,1.0
28,133707,182261,1,3,10,NaN
36,3378084,127629,21,1,10,20.0
49,2161799,87345,7,0,10,24.0
51,1739741,118405,48,2,10,3.0


Resultado: 40.578 de 478.967 pedidos (8,47 %)


**Explicación:** muestra cuántos pedidos se registraron a las 10:00, una hora de la franja de la mañana.

## Requerimiento 4 · pedidos con `order_dow` igual a 0 (orders)

In [23]:
ej4 = orders.loc[orders["order_dow"] == 0]
display(ej4.head())
print(f"Resultado: {fmt(ej4.shape[0])} de {fmt(orders.shape[0])} pedidos ({fmt(100 * ej4.shape[0] / orders.shape[0], 2)} %)")

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
3,1768857,82516,56,0,20,10.0
5,1843871,20084,5,0,11,15.0
7,2521283,97711,2,0,18,22.0
23,3406282,19356,21,0,0,14.0
25,1895419,169793,2,0,11,8.0


Resultado: 84.090 de 478.967 pedidos (17,56 %)


**Explicación:** `order_dow` codifica el día de la semana (0 a 6). El filtro aísla los pedidos del día codificado como 0; sirve para comparar el volumen de ese día con el de los demás.

## Requerimiento 5 · pedidos con `order_number` superior a 10 (orders)

In [24]:
ej5 = orders.loc[orders["order_number"] > 10]
display(ej5.head())
print(f"Resultado: {fmt(ej5.shape[0])} de {fmt(orders.shape[0])} pedidos ({fmt(100 * ej5.shape[0] / orders.shape[0], 2)} %)")

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
3,1768857,82516,56,0,20,10.0
6,2511640,148233,40,3,8,2.0
8,2266494,34954,40,5,15,4.0
10,1162566,43738,33,4,11,2.0


Resultado: 244.072 de 478.967 pedidos (50,96 %)


**Explicación:** un `order_number` alto indica que el usuario ya había realizado muchas compras; son pedidos de clientes recurrentes.

## Requerimiento 6 · pedidos con `days_since_prior_order` superior a 15 (orders)

In [25]:
ej6 = orders.loc[orders["days_since_prior_order"] > 15]
display(ej6.head())
print(f"Resultado: {fmt(ej6.shape[0])} de {fmt(orders.shape[0])} pedidos ({fmt(100 * ej6.shape[0] / orders.shape[0], 2)} %)")
print(f"Nota: {fmt(orders['days_since_prior_order'].isna().sum())} pedidos tienen el dato ausente y la comparación `> 15` no los selecciona.")

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
4,3007858,196724,2,4,12,17.0
7,2521283,97711,2,0,18,22.0
14,2843219,74699,4,6,11,30.0
26,1844247,122558,4,6,18,30.0


Resultado: 110.706 de 478.967 pedidos (23,11 %)
Nota: 28.819 pedidos tienen el dato ausente y la comparación `> 15` no los selecciona.


**Explicación:** son pedidos hechos más de 15 días después del anterior (compra poco frecuente). Los valores ausentes quedan fuera del filtro porque `NaN > 15` es falso.

## Requerimiento 7 · registros con `reordered` igual a 1 (order_products)

In [26]:
ej7 = order_products.loc[order_products["reordered"] == 1]
display(ej7.head())
print(f"Resultado: {fmt(ej7.shape[0])} de {fmt(order_products.shape[0])} registros ({fmt(100 * ej7.shape[0] / order_products.shape[0], 2)} %)")

,order_id,product_id,add_to_cart_order,reordered
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
3,491251,8670,35.0,1
4,2571142,1940,5.0,1
5,2456893,21616,4.0,1


Resultado: 2.683.838 de 4.545.007 registros (59,05 %)


**Explicación:** selecciona los artículos que el usuario ya había comprado antes (recompras). Su proporción indica la lealtad hacia ciertos productos.

## Requerimiento 8 · productos en la primera posición del carrito (`add_to_cart_order` igual a 1)

In [27]:
ej8 = order_products.loc[order_products["add_to_cart_order"] == 1]
display(ej8.head())
print(f"Resultado: {fmt(ej8.shape[0])} registros")
print(f"Pedidos distintos en order_products: {fmt(order_products['order_id'].nunique())}")

,order_id,product_id,add_to_cart_order,reordered
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
14,1961225,37553,1.0,1
16,639939,10017,1.0,1
23,750040,8518,1.0,0


Resultado: 450.046 registros


Pedidos distintos en order_products: 450.046


**Explicación:** se obtiene el primer artículo que se agregó a cada carrito. Idealmente hay uno por pedido; si la cifra difiere del número de pedidos distintos, se debe a pedidos con la posición ausente.

## Filtros con condiciones simultáneas

### Filtro A · pedidos a las 10 horas **y** con `order_number` superior a 10

In [28]:
filtro_a = orders.loc[(orders["order_hour_of_day"] == 10) & (orders["order_number"] > 10)]
display(filtro_a.head())
print(f"Solo hora 10:            {fmt(ej3.shape[0])}")
print(f"Solo order_number > 10:  {fmt(ej5.shape[0])}")
print(f"Ambas condiciones (&):   {fmt(filtro_a.shape[0])}")

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
19,294269,107523,29,2,10,1.0
36,3378084,127629,21,1,10,20.0
51,1739741,118405,48,2,10,3.0
92,60724,110808,78,0,10,2.0
140,3156095,113920,14,2,10,2.0


Solo hora 10:            40.578
Solo order_number > 10:  244.072
Ambas condiciones (&):   21.537


**Explicación:** con `&` solo se conservan las filas que cumplen **ambas** condiciones, por lo que el resultado nunca supera a ninguno de los dos filtros individuales.

### Filtro B · productos del departamento 1 **y** con `aisle_id` superior a 50

In [29]:
filtro_b = products.loc[(products["department_id"] == 1) & (products["aisle_id"] > 50)]
display(filtro_b.head())
print(f"Solo department_id == 1: {fmt(ej2.shape[0])}")
print(f"Ambas condiciones (&):   {fmt(filtro_b.shape[0])}")

,product_id,product_name,aisle_id,department_id
7,8,Cut Russet Potatoes Steam N' Mash,116,1
11,12,Chocolate Fudge Layer Cake,119,1
17,18,Pizza for One Suprema Frozen Pizza,79,1
45,46,Raisin Cinnamon Bagels 5 count,58,1
127,128,Organic Wild Blueberries,116,1


Solo department_id == 1: 4.008
Ambas condiciones (&):   1.618


**Explicación:** el resultado es un subconjunto aún más específico del departamento 1: solo los productos que además están en pasillos con identificador mayor a 50.

## Preguntas de análisis

In [30]:
display(Markdown(f"""
**¿Qué diferencia existe entre seleccionar columnas y filtrar filas?**
Seleccionar columnas reduce el *ancho* de la tabla (conserva todas las filas, p. ej. el Requerimiento 1: {fmt(ej1.shape[0])} filas, {ej1.shape[1]} columnas). Filtrar filas reduce el *alto* (conserva las columnas que cumplen una condición, p. ej. el Requerimiento 2: {fmt(ej2.shape[0])} filas, {ej2.shape[1]} columnas).

**¿Qué ventajas ofrece `loc[]`?**
Permite combinar en una sola expresión **filas (por condición o etiqueta) y columnas (por nombre)**, es explícito y legible, y evita ambigüedades al asignar valores. Ejemplo: `products.loc[products["department_id"] == 1, ["product_id", "product_name"]]`.

**¿Qué ocurre con la cantidad de registros cuando se aplican varias condiciones?**
Con `&` disminuye o se mantiene: en el Filtro A se pasó de {fmt(ej3.shape[0])} (solo hora 10) y {fmt(ej5.shape[0])} (solo `order_number` > 10) a {fmt(filtro_a.shape[0])}; en el Filtro B se pasó de {fmt(ej2.shape[0])} a {fmt(filtro_b.shape[0])}.

**¿Por qué es importante usar correctamente los operadores lógicos?**
En Pandas se usan `&`, `|` y `~`, no `and`, `or` y `not`, que fallan con Series. Además, cada comparación debe ir entre paréntesis por la precedencia de operadores. Un operador incorrecto cambia el conjunto seleccionado y, con él, todos los indicadores posteriores (`&` exige ambas condiciones, `|` basta con una).
"""))


**¿Qué diferencia existe entre seleccionar columnas y filtrar filas?**
Seleccionar columnas reduce el *ancho* de la tabla (conserva todas las filas, p. ej. el Requerimiento 1: 49.694 filas, 2 columnas). Filtrar filas reduce el *alto* (conserva las columnas que cumplen una condición, p. ej. el Requerimiento 2: 4.008 filas, 4 columnas).

**¿Qué ventajas ofrece `loc[]`?**
Permite combinar en una sola expresión **filas (por condición o etiqueta) y columnas (por nombre)**, es explícito y legible, y evita ambigüedades al asignar valores. Ejemplo: `products.loc[products["department_id"] == 1, ["product_id", "product_name"]]`.

**¿Qué ocurre con la cantidad de registros cuando se aplican varias condiciones?**
Con `&` disminuye o se mantiene: en el Filtro A se pasó de 40.578 (solo hora 10) y 244.072 (solo `order_number` > 10) a 21.537; en el Filtro B se pasó de 4.008 a 1.618.

**¿Por qué es importante usar correctamente los operadores lógicos?**
En Pandas se usan `&`, `|` y `~`, no `and`, `or` y `not`, que fallan con Series. Además, cada comparación debe ir entre paréntesis por la precedencia de operadores. Un operador incorrecto cambia el conjunto seleccionado y, con él, todos los indicadores posteriores (`&` exige ambas condiciones, `|` basta con una).


---
# Momento 5 · Obtención e interpretación de indicadores básicos

Primero distingo **registros**, **pedidos** y **usuarios**: el número de filas no equivale al número de clientes.

In [31]:
n_filas_orders = orders.shape[0]
n_pedidos_unicos = orders["order_id"].nunique()
n_usuarios = orders["user_id"].nunique()
n_filas_op = order_products.shape[0]
n_pedidos_en_op = order_products["order_id"].nunique()
n_productos_en_op = order_products["product_id"].nunique()

distincion = pd.DataFrame(
    {
        "concepto": [
            "Registros (filas) en orders",
            "Pedidos distintos (order_id) en orders",
            "Usuarios distintos (user_id) en orders",
            "Registros (filas) en order_products",
            "Pedidos distintos presentes en order_products",
            "Productos distintos presentes en order_products",
        ],
        "valor": [
            n_filas_orders, n_pedidos_unicos, n_usuarios,
            n_filas_op, n_pedidos_en_op, n_productos_en_op,
        ],
    }
)
distincion["valor"] = distincion["valor"].map(fmt)
display(distincion)

print(f"Promedio de pedidos por usuario: {fmt(n_pedidos_unicos / n_usuarios, 2)}")
print(f"Promedio de artículos por pedido: {fmt(n_filas_op / n_pedidos_en_op, 2)}")

,concepto,valor
0,Registros (filas) en orders,478.967
1,Pedidos distintos (order_id) en orders,478.952
2,Usuarios distintos (user_id) en orders,157.437
3,Registros (filas) en order_products,4.545.007
4,Pedidos distintos presentes en order_products,450.046
5,Productos distintos presentes en order_products,45.573


Promedio de pedidos por usuario: 3,04
Promedio de artículos por pedido: 10,10


## 5.1 Cálculo de cada indicador

In [32]:
# 1. Registros de pedidos
v_registros_pedidos = orders.shape[0]
# 2. Registros de productos
v_registros_productos = products.shape[0]
# 3. Promedio de order_number
v_prom_order_number = orders["order_number"].mean()
# 4. Promedio de days_since_prior_order (mean ignora NaN)
v_prom_dias = orders["days_since_prior_order"].mean()
n_dias_disp = orders["days_since_prior_order"].count()
# 5. Registros con reordered == 1 (filtrado y conteo)
v_reordered_1 = order_products.loc[order_products["reordered"] == 1].shape[0]
# 6. Suma de reordered
v_suma_reordered = order_products["reordered"].sum()

print("Registros de pedidos:      ", fmt(v_registros_pedidos))
print("Registros de productos:    ", fmt(v_registros_productos))
print("Promedio order_number:     ", fmt(v_prom_order_number, 2))
print("Promedio days_since_prior: ", fmt(v_prom_dias, 2), f"(sobre {fmt(n_dias_disp)} valores disponibles)")
print("Registros reordered == 1:  ", fmt(v_reordered_1))
print("Suma de reordered:         ", fmt(v_suma_reordered))
print("¿Coinciden filtrado+conteo y suma?", v_reordered_1 == v_suma_reordered)

Registros de pedidos:       478.967
Registros de productos:     49.694
Promedio order_number:      17,16
Promedio days_since_prior:  11,10 (sobre 450.148 valores disponibles)
Registros reordered == 1:   2.683.838
Suma de reordered:          2.683.838
¿Coinciden filtrado+conteo y suma? True


In [33]:
desc_hora = orders["order_hour_of_day"].describe()
print("describe() de order_hour_of_day")
display(desc_hora.to_frame())

describe() de order_hour_of_day


,order_hour_of_day
count,478967.000000
mean,13.446676
std,4.224986
min,0.000000
25%,10.000000
50%,13.000000
75%,16.000000
max,23.000000


In [34]:
desc_carrito = order_products["add_to_cart_order"].describe()
print("describe() de add_to_cart_order")
display(desc_carrito.to_frame())

describe() de add_to_cart_order


,add_to_cart_order
count,4.544171e+06
mean,8.351669e+00
std,7.080368e+00
min,1.000000e+00
25%,3.000000e+00
50%,6.000000e+00
75%,1.100000e+01
max,6.400000e+01


## 5.2 Tabla de indicadores: pregunta, valor e interpretación

In [35]:
prop_reorder = v_reordered_1 / order_products.shape[0]
faltan_carrito = order_products["add_to_cart_order"].isna().sum()
articulos_por_pedido = order_products.groupby("order_id").size()
pedidos_con_ausente = order_products.loc[order_products["add_to_cart_order"].isna(), "order_id"].unique()

interpretacion_horas = (
    f"Los pedidos se hacen entre las {desc_hora['min']:.0f} y las {desc_hora['max']:.0f} h; "
    f"el promedio es {fmt(desc_hora['mean'], 2)} h y la mediana {fmt(desc_hora['50%'], 0)} h. "
    f"El 50 % central de los pedidos ocurre entre las {fmt(desc_hora['25%'], 0)} y las {fmt(desc_hora['75%'], 0)} h."
)
interpretacion_carrito = (
    f"Se evaluaron {fmt(desc_carrito['count'])} de {fmt(order_products.shape[0])} registros "
    f"({fmt(faltan_carrito)} ausentes). El promedio de posición es {fmt(desc_carrito['mean'], 2)}, "
    f"la mediana {fmt(desc_carrito['50%'], 0)} y el máximo {fmt(desc_carrito['max'], 0)}: "
    f"la mitad de los artículos se agregó en las primeras {fmt(desc_carrito['50%'], 0)} posiciones. "
    f"Por pedido, la mediana es de {fmt(articulos_por_pedido.median(), 0)} artículos y solo el "
    f"{fmt(100 * (articulos_por_pedido > 30).mean(), 1)} % supera los 30, así que la mayoría de los carritos es pequeña. "
    f"Los ausentes están en {fmt(len(pedidos_con_ausente))} pedidos de {fmt(articulos_por_pedido.loc[pedidos_con_ausente].min())} "
    f"artículos o más, por eso el máximo registrado ({fmt(desc_carrito['max'], 0)}) es menor que el pedido más grande "
    f"({fmt(articulos_por_pedido.max())} artículos)."
)

tabla_indicadores = pd.DataFrame(
    [
        (
            "¿Cuántos registros de pedidos contiene el archivo?",
            fmt(v_registros_pedidos),
            f"Hay {fmt(v_registros_pedidos)} filas en orders, pero corresponden a {fmt(n_pedidos_unicos)} pedidos distintos "
            f"y {fmt(n_usuarios)} usuarios distintos; las filas no equivalen a clientes.",
        ),
        (
            "¿Cuántos registros de productos contiene el catálogo?",
            fmt(v_registros_productos),
            f"El catálogo tiene {fmt(v_registros_productos)} filas y {fmt(products['product_id'].nunique())} product_id distintos.",
        ),
        (
            "¿Cuál es el promedio de order_number?",
            fmt(v_prom_order_number, 2),
            f"En promedio un pedido es el número {fmt(v_prom_order_number, 1)} de su usuario, "
            f"lo que indica una base de clientes con historial de compra repetida.",
        ),
        (
            "¿Cuál es el promedio de days_since_prior_order entre los valores disponibles?",
            fmt(v_prom_dias, 2),
            f"Entre pedidos consecutivos pasan en promedio {fmt(v_prom_dias, 1)} días (sobre {fmt(n_dias_disp)} valores; "
            f"se excluyen {fmt(orders.shape[0] - n_dias_disp)} ausentes, que son primeros pedidos).",
        ),
        (
            "¿Cuántos registros de artículos tienen reordered igual a 1?",
            fmt(v_reordered_1),
            f"Son {fmt(prop_reorder * 100, 2)} % de los {fmt(order_products.shape[0])} artículos registrados: "
            f"productos que el usuario ya había comprado.",
        ),
        (
            "¿Cuál es la suma de la columna reordered?",
            fmt(v_suma_reordered),
            "Como reordered solo toma los valores 0 y 1, su suma coincide con el conteo de registros con reordered igual a 1.",
        ),
        (
            "¿Qué características estadísticas presenta order_hour_of_day?",
            f"media {fmt(desc_hora['mean'], 2)}; mediana {fmt(desc_hora['50%'], 0)}; rango {desc_hora['min']:.0f}–{desc_hora['max']:.0f}",
            interpretacion_horas,
        ),
        (
            "¿Qué características estadísticas presenta add_to_cart_order?",
            f"media {fmt(desc_carrito['mean'], 2)}; mediana {fmt(desc_carrito['50%'], 0)}; máx. {fmt(desc_carrito['max'], 0)}",
            interpretacion_carrito,
        ),
    ],
    columns=["Pregunta analizada", "Valor obtenido", "Interpretación"],
)

with pd.option_context("display.max_colwidth", 600):
    display(tabla_indicadores)

,Pregunta analizada,Valor obtenido,Interpretación
0,¿Cuántos registros de pedidos contiene el archivo?,478.967,"Hay 478.967 filas en orders, pero corresponden a 478.952 pedidos distintos y 157.437 usuarios distintos; las filas no equivalen a clientes."
1,¿Cuántos registros de productos contiene el catálogo?,49.694,El catálogo tiene 49.694 filas y 49.694 product_id distintos.
2,¿Cuál es el promedio de order_number?,"17,16","En promedio un pedido es el número 17,2 de su usuario, lo que indica una base de clientes con historial de compra repetida."
3,¿Cuál es el promedio de days_since_prior_order entre los valores disponibles?,"11,10","Entre pedidos consecutivos pasan en promedio 11,1 días (sobre 450.148 valores; se excluyen 28.819 ausentes, que son primeros pedidos)."
4,¿Cuántos registros de artículos tienen reordered igual a 1?,2.683.838,"Son 59,05 % de los 4.545.007 artículos registrados: productos que el usuario ya había comprado."
5,¿Cuál es la suma de la columna reordered?,2.683.838,"Como reordered solo toma los valores 0 y 1, su suma coincide con el conteo de registros con reordered igual a 1."
6,¿Qué características estadísticas presenta order_hour_of_day?,"media 13,45; mediana 13; rango 0–23","Los pedidos se hacen entre las 0 y las 23 h; el promedio es 13,45 h y la mediana 13 h. El 50 % central de los pedidos ocurre entre las 10 y las 16 h."
7,¿Qué características estadísticas presenta add_to_cart_order?,"media 8,35; mediana 6; máx. 64","Se evaluaron 4.544.171 de 4.545.007 registros (836 ausentes). El promedio de posición es 8,35, la mediana 6 y el máximo 64: la mitad de los artículos se agregó en las primeras 6 posiciones. Por pedido, la mediana es de 8 artículos y solo el 2,1 % supera los 30, así que la mayoría de los carritos es pequeña. Los ausentes están en 70 pedidos de 65 artículos o más, por eso el máximo registrado (64) es menor que el pedido más grande (127 artículos)."


## 5.3 Complemento: distribución de pedidos por hora y por día

In [36]:
pedidos_por_hora = orders["order_hour_of_day"].value_counts().sort_index()
pedidos_por_dia = orders["order_dow"].value_counts().sort_index()

hora_pico = pedidos_por_hora.idxmax()
dia_pico = pedidos_por_dia.idxmax()
dia_bajo = pedidos_por_dia.idxmin()

display(pedidos_por_hora.rename("pedidos").to_frame())
display(pedidos_por_dia.rename("pedidos").to_frame())

print(f"Hora con más pedidos: {hora_pico} h ({fmt(pedidos_por_hora.max())} pedidos)")
print(f"Valor de order_dow con más pedidos: {dia_pico} ({fmt(pedidos_por_dia.max())}); con menos: {dia_bajo} ({fmt(pedidos_por_dia.min())})")

,pedidos
order_hour_of_day,
0,3180
1,1763
2,1004
3,770
4,765
5,1371
6,4215
7,13043
8,25024


,pedidos
order_dow,
0,84090
1,82185
2,65833
3,60912
4,59810
5,63488
6,62649


Hora con más pedidos: 10 h (40.578 pedidos)
Valor de order_dow con más pedidos: 0 (84.090); con menos: 4 (59.810)


---
# Momento 6 · Resolución del problema y presentación del producto

Síntesis individual que responde la pregunta orientadora. Las cifras se calculan con los resultados de los momentos anteriores.

## 6.1 Tres preguntas de análisis y sus respuestas

In [37]:
franja_alta = pedidos_por_hora[pedidos_por_hora >= pedidos_por_hora.max() * 0.8]
rango_alta = f"{franja_alta.index.min()} y {franja_alta.index.max()} h"

display(Markdown(f"""
**1. ¿A qué horas y en qué días se concentran los pedidos?**
La hora con más pedidos es las **{hora_pico} h** ({fmt(pedidos_por_hora.max())} pedidos). Las horas con al menos el 80 % del volumen máximo van entre las {rango_alta}. Por día, el valor de `order_dow` con más pedidos es **{dia_pico}** ({fmt(pedidos_por_dia.max())}) y el de menos es **{dia_bajo}** ({fmt(pedidos_por_dia.min())}).

**2. ¿Con qué frecuencia vuelven a comprar los usuarios?**
Entre pedidos consecutivos pasan en promedio **{fmt(v_prom_dias, 2)} días**, y el pedido promedio es el número **{fmt(v_prom_order_number, 2)}** de cada usuario. En este archivo, los {fmt(n_usuarios)} usuarios suman {fmt(n_pedidos_unicos)} pedidos distintos, unos {fmt(n_pedidos_unicos / n_usuarios, 1)} por usuario; como el `order_number` promedio es mucho mayor, el archivo solo contiene una parte del historial de cada cliente.

**3. ¿Qué proporción de artículos son recompras y cuántos se agregan primero al carrito?**
**{fmt(prop_reorder * 100, 2)} %** de los artículos son recompras ({fmt(v_reordered_1)} de {fmt(order_products.shape[0])}). Hay **{fmt(ej8.shape[0])}** registros en primera posición del carrito frente a {fmt(n_pedidos_en_op)} pedidos con artículos.
"""))


**1. ¿A qué horas y en qué días se concentran los pedidos?**
La hora con más pedidos es las **10 h** (40.578 pedidos). Las horas con al menos el 80 % del volumen máximo van entre las 9 y 16 h. Por día, el valor de `order_dow` con más pedidos es **0** (84.090) y el de menos es **4** (59.810).

**2. ¿Con qué frecuencia vuelven a comprar los usuarios?**
Entre pedidos consecutivos pasan en promedio **11,10 días**, y el pedido promedio es el número **17,16** de cada usuario. En este archivo, los 157.437 usuarios suman 478.952 pedidos distintos, unos 3,0 por usuario; como el `order_number` promedio es mucho mayor, el archivo solo contiene una parte del historial de cada cliente.

**3. ¿Qué proporción de artículos son recompras y cuántos se agregan primero al carrito?**
**59,05 %** de los artículos son recompras (2.683.838 de 4.545.007). Hay **450.046** registros en primera posición del carrito frente a 450.046 pedidos con artículos.


## 6.2 Principales hallazgos sobre pedidos y productos

In [38]:
display(Markdown(f"""
**Pedidos**
- `orders` tiene {fmt(orders.shape[0])} registros, que corresponden a {fmt(n_pedidos_unicos)} pedidos distintos de {fmt(n_usuarios)} usuarios.
- Los pedidos se registran a lo largo de las 24 horas; la mediana está en las {fmt(desc_hora['50%'], 0)} h.
- Los pedidos con `order_number` superior a 10 son {fmt(ej5.shape[0])} ({fmt(100 * ej5.shape[0] / orders.shape[0], 1)} %), lo que muestra la presencia de clientes recurrentes.
- Los pedidos realizados más de 15 días después del anterior son {fmt(ej6.shape[0])} ({fmt(100 * ej6.shape[0] / orders.shape[0], 1)} %).

**Productos y artículos**
- El catálogo tiene {fmt(products.shape[0])} productos, clasificados en {fmt(aisles.shape[0])} pasillos y {fmt(departments.shape[0])} departamentos.
- `order_products` registra {fmt(order_products.shape[0])} artículos, con {fmt(order_products.shape[0] / n_pedidos_en_op, 1)} artículos por pedido en promedio.
- El departamento 1 reúne {fmt(ej2.shape[0])} productos ({fmt(100 * ej2.shape[0] / products.shape[0], 1)} % del catálogo); {fmt(filtro_b.shape[0])} de ellos están en pasillos con `aisle_id` superior a 50.
"""))


**Pedidos**
- `orders` tiene 478.967 registros, que corresponden a 478.952 pedidos distintos de 157.437 usuarios.
- Los pedidos se registran a lo largo de las 24 horas; la mediana está en las 13 h.
- Los pedidos con `order_number` superior a 10 son 244.072 (51,0 %), lo que muestra la presencia de clientes recurrentes.
- Los pedidos realizados más de 15 días después del anterior son 110.706 (23,1 %).

**Productos y artículos**
- El catálogo tiene 49.694 productos, clasificados en 134 pasillos y 21 departamentos.
- `order_products` registra 4.545.007 artículos, con 10,1 artículos por pedido en promedio.
- El departamento 1 reúne 4.008 productos (8,1 % del catálogo); 1.618 de ellos están en pasillos con `aisle_id` superior a 50.


## 6.3 Cómo la indexación y los filtros permitieron consultar información relevante

In [39]:
display(Markdown(f"""
- La **selección de columnas** con `loc[:, [...]]` permitió ver solo `product_id` y `product_name` sin cargar el resto de campos.
- Los **filtros con una condición** respondieron preguntas puntuales: por ejemplo, {fmt(ej3.shape[0])} pedidos a las 10 h, {fmt(ej7.shape[0])} recompras o {fmt(ej6.shape[0])} pedidos con más de 15 días de espera.
- Los **filtros con varias condiciones** acotaron la búsqueda: de {fmt(ej5.shape[0])} pedidos con `order_number` > 10 se pasó a {fmt(filtro_a.shape[0])} al exigir además la hora 10.
- Los **indicadores** (`mean`, `sum`, `describe`, `count`) resumieron millones de filas en pocas cifras interpretables.
"""))


- La **selección de columnas** con `loc[:, [...]]` permitió ver solo `product_id` y `product_name` sin cargar el resto de campos.
- Los **filtros con una condición** respondieron preguntas puntuales: por ejemplo, 40.578 pedidos a las 10 h, 2.683.838 recompras o 110.706 pedidos con más de 15 días de espera.
- Los **filtros con varias condiciones** acotaron la búsqueda: de 244.072 pedidos con `order_number` > 10 se pasó a 21.537 al exigir además la hora 10.
- Los **indicadores** (`mean`, `sum`, `describe`, `count`) resumieron millones de filas en pocas cifras interpretables.


## 6.4 Limitaciones identificadas en los conjuntos de datos

In [40]:
lim = []
for _, r in ausentes[ausentes["ausentes"] > 0].iterrows():
    lim.append(
        f"- **Valores ausentes** en `{r.DataFrame}.{r.columna}`: {fmt(r.ausentes)} registros ({fmt(r['% ausente'], 2)} %)."
    )
dup_orders = int(orders.duplicated(subset=["order_id"]).sum())
if dup_orders:
    lim.append(
        f"- **`order_id` repetidos** en `orders`: {fmt(dup_orders)}. Por eso las filas ({fmt(orders.shape[0])}) superan a los pedidos distintos ({fmt(n_pedidos_unicos)})."
    )
dup_op = int(order_products.duplicated(subset=["order_id", "product_id"]).sum())
if dup_op:
    lim.append(f"- **Pares `order_id`-`product_id` repetidos** en `order_products`: {fmt(dup_op)}.")
dup_filas = {n: int(df.duplicated().sum()) for n, df in datasets.items() if df.duplicated().sum() > 0}
for n, v in dup_filas.items():
    lim.append(f"- **Filas completamente duplicadas** en `{n}`: {fmt(v)}.")
if not lim:
    lim.append("- No se detectaron ausentes ni duplicados en esta exploración.")

lim += [
    "- Los datos describen **pedidos y artículos**, no el precio, la cantidad por artículo ni datos del cliente (edad, ubicación), por lo que no permiten analizar gasto ni perfiles.",
    "- `order_dow` es un código numérico; sin un diccionario de datos no se puede asegurar a qué día corresponde cada valor.",
    "- Esta fase **no limpia** los datos: los indicadores se calcularon sobre los datos tal como llegaron, incluidos ausentes y repetidos.",
]
display(Markdown("\n".join(lim)))

- **Valores ausentes** en `orders.days_since_prior_order`: 28.819 registros (6,02 %).
- **Valores ausentes** en `products.product_name`: 1.258 registros (2,53 %).
- **Valores ausentes** en `order_products.add_to_cart_order`: 836 registros (0,02 %).
- **`order_id` repetidos** en `orders`: 15. Por eso las filas (478.967) superan a los pedidos distintos (478.952).
- **Filas completamente duplicadas** en `orders`: 15.
- Los datos describen **pedidos y artículos**, no el precio, la cantidad por artículo ni datos del cliente (edad, ubicación), por lo que no permiten analizar gasto ni perfiles.
- `order_dow` es un código numérico; sin un diccionario de datos no se puede asegurar a qué día corresponde cada valor.
- Esta fase **no limpia** los datos: los indicadores se calcularon sobre los datos tal como llegaron, incluidos ausentes y repetidos.

## 6.5 Tres conclusiones sustentadas en los resultados

In [41]:
calificativo = "la mayoría" if prop_reorder > 0.5 else "una parte importante"
pct_franja = 100 * franja_alta.sum() / orders.shape[0]
segunda_hora = pedidos_por_hora.nlargest(2).index[1]

display(Markdown(f"""
1. **La compra se concentra en una franja del día.** Entre las {rango_alta} se registra el {fmt(pct_franja, 1)} % de los pedidos. La hora con más pedidos es las {hora_pico} h ({fmt(pedidos_por_hora.max())}), seguida muy de cerca por las {segunda_hora} h ({fmt(pedidos_por_hora[segunda_hora])}), y la mediana de `order_hour_of_day` es {fmt(desc_hora['50%'], 0)} h. Esto sugiere reforzar la operación en esa franja.
2. **Los clientes son recurrentes y compran con regularidad.** El promedio de `order_number` es {fmt(v_prom_order_number, 1)} (en este archivo aparecen unos {fmt(n_pedidos_unicos / n_usuarios, 1)} pedidos por usuario porque solo recoge parte de su historial) y el tiempo medio entre pedidos es de {fmt(v_prom_dias, 1)} días (mediana {fmt(orders['days_since_prior_order'].median(), 0)} días).
3. **Los hábitos de compra están marcados por la recompra.** {calificativo.capitalize()} de los artículos ({fmt(prop_reorder * 100, 1)} %) son productos ya comprados antes, lo que muestra fidelidad hacia ciertos productos y valida usar `reordered` como indicador de hábito.

**Respuesta a la pregunta orientadora:** con Pandas se pudo importar los CSV, conocer su estructura y calidad, consultar información puntual con `loc[]` y filtros lógicos, y resumir el comportamiento de los clientes con indicadores básicos. La exploración también mostró que antes de un análisis más profundo se deben tratar los valores ausentes y repetidos documentados en 6.4.
"""))


1. **La compra se concentra en una franja del día.** Entre las 9 y 16 h se registra el 64,9 % de los pedidos. La hora con más pedidos es las 10 h (40.578), seguida muy de cerca por las 11 h (40.032), y la mediana de `order_hour_of_day` es 13 h. Esto sugiere reforzar la operación en esa franja.
2. **Los clientes son recurrentes y compran con regularidad.** El promedio de `order_number` es 17,2 (en este archivo aparecen unos 3,0 pedidos por usuario porque solo recoge parte de su historial) y el tiempo medio entre pedidos es de 11,1 días (mediana 7 días).
3. **Los hábitos de compra están marcados por la recompra.** La mayoría de los artículos (59,1 %) son productos ya comprados antes, lo que muestra fidelidad hacia ciertos productos y valida usar `reordered` como indicador de hábito.

**Respuesta a la pregunta orientadora:** con Pandas se pudo importar los CSV, conocer su estructura y calidad, consultar información puntual con `loc[]` y filtros lógicos, y resumir el comportamiento de los clientes con indicadores básicos. La exploración también mostró que antes de un análisis más profundo se deben tratar los valores ausentes y repetidos documentados en 6.4.


## 6.6 Verificación final del notebook

In [42]:
verificaciones = {
    "Los cinco DataFrames están cargados": all(len(df) > 0 for df in datasets.values()),
    "Las dimensiones coinciden con el enunciado": all(df.shape == esperado[n] for n, df in datasets.items()),
    "reordered: filtrado y conteo igual a suma": v_reordered_1 == v_suma_reordered,
    "Pedidos distintos <= registros en orders": n_pedidos_unicos <= n_filas_orders,
    "Los datos de orders no fueron modificados (6 columnas)": orders.shape[1] == 6,
}
for texto, ok in verificaciones.items():
    print(("OK   " if ok else "REVISAR ") + texto)

OK   Los cinco DataFrames están cargados
OK   Las dimensiones coinciden con el enunciado
OK   reordered: filtrado y conteo igual a suma
OK   Pedidos distintos <= registros en orders
OK   Los datos de orders no fueron modificados (6 columnas)
